<center><p float="center">
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<h1><center> Real-Time Retail Feedback Intelligence
 </center></h1>


### **Business Context**
ChicStyle is a growing fashion retail platform that receives a large spike in customer reviews during festive seasons and holiday sales. These reviews contain both praise and urgent complaints about fit, sizing, product quality, delivery, comfort, color accuracy, and styling. During peak periods, delayed review analysis can lead to unresolved customer frustration, lower trust, lost repeat purchases, and damage to brand reputation.

This problem is important because customer feedback is high-volume, time-sensitive, and nuanced. Traditional rule-based or limited NLP systems often miss mixed feedback such as "the fit is great but the color was not as shown." A Generative AI pipeline can read reviews quickly, identify the sentiment and topic, summarize the issue, suggest a customer response, and surface retail insights for business teams.

### **Objective**
The goal is to build a low-code, real-time retail feedback intelligence workflow that uses Generative AI to:

* classify each review into a business-relevant category;
* detect sentiment as Positive, Negative, or Neutral;
* create a concise review summary;
* generate a personalized customer-facing response;
* extract actionable retail insights;
* predict whether the customer would recommend the product; and
* compare Zero-Shot, Few-Shot, and Chain-of-Thought prompting strategies using an LLM-as-Judge framework.

### **Dataset Used for the Notebook**
The dataset contains women's clothing e-commerce reviews. Each row represents one customer review and includes product and customer metadata such as `Clothing.ID`, `Age`, `Title`, `Review.Text`, `Rating`, `Recommended.IND`, `Positive.Feedback.Count`, `Division.Name`, `Department.Name`, and `Class.Name`. The key modeling feature is `Review.Text`; `Rating` and `Recommended.IND` are useful for validation, exploratory analysis, and recommendation evaluation.


### **Installing and Importing Necessary Libraries**
First, let's set up the environment by installing the required Python libraries.

In [ ]:
# Install the required libraries for the project
# --- Installation of Required Libraries ---
!pip install pandas matplotlib seaborn scikit-learn
!pip install -q -U google-genai
!pip install openai
!pip install wordcloud
!pip install plotly


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.5/260.5 kB 8.3 MB/s eta 0:00:00


In [ ]:
# Import the required libraries for the project
# --- Core Utilities ---
import time
import json
import re
import numpy as np
from typing import Dict, List, Optional, Any, Tuple

# --- Data Handling ---
import pandas as pd
from google.colab import userdata

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

# --- Machine Learning Metrics ---
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# --- Progress Bar ---
from tqdm import tqdm

# --- OpenAI Client ---
import openai

# Set global plot style
sns.set_style("whitegrid")

### **Data Loading**
### Loading and Understanding the Data


In [ ]:
### Loading and Understanding the Data
# Load the dataset. In Colab, upload the CSV file to the session files area
# or change DATA_PATH to the exact location in Google Drive.

DATA_PATH = "Dataset - Real-Time Retail Feedback Intelligence.csv"
df = pd.read_csv(DATA_PATH, sep=";", index_col=0)
display(df)


,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
1,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
2,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
3,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
4,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
5,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses
...,...,...,...,...,...,...,...,...,...,...
23482,1104,34,Great dress for many occasions,I was very happy to snag this dress at such a ...,5,1,0,General Petite,Dresses,Dresses
23483,862,48,Wish it was made of cotton,"It reminds me of maternity clothes. soft, stre...",3,1,0,General Petite,Tops,Knits
23484,1104,31,"Cute, but see through","This fit well, but the top was very see throug...",3,0,1,General Petite,Dresses,Dresses
23485,1084,28,"Very cute dress, perfect for summer parties an...",I bought this dress for a wedding i have this ...,3,1,2,General,Dresses,Dresses


### **Data Overview**

In [ ]:
# Get a summary of the dataset to check data types and non-null counts
print("\nDataset Info:")
df.info()

Dataset Head:


,Clothing.ID,Age,Title,Review.Text,Rating,Recommended.IND,Positive.Feedback.Count,Division.Name,Department.Name,Class.Name
1,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
2,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
3,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
4,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
5,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


### **Sanity checks**

In [ ]:
# Check for the total number of missing values in each column
print("\nMissing Values:")
print(df.isnull().sum())



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 23486 entries, 1 to 23486
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Clothing.ID              23486 non-null  int64 
 1   Age                      23486 non-null  int64 
 2   Title                    19676 non-null  object
 3   Review.Text              22641 non-null  object
 4   Rating                   23486 non-null  int64 
 5   Recommended.IND          23486 non-null  int64 
 6   Positive.Feedback.Count  23486 non-null  int64 
 7   Division.Name            23472 non-null  object
 8   Department.Name          23472 non-null  object
 9   Class.Name               23472 non-null  object
dtypes: int64(5), object(5)
memory usage: 2.0+ MB


### **Data Cleaning and Preprocessing**

**Think about it:** The `Review.Text` column is the most critical feature for the Generative AI model. Rows where this text is missing should be removed from the modeling dataset because the LLM cannot infer sentiment, category, summary, recommendation intent, or customer response without the actual review content. Other missing fields, such as `Title`, can be retained because they are optional metadata and are not required for the core prompt.


In [ ]:
df = df.dropna(subset=["Review.Text"]).copy()
df["Review.Text"] = df["Review.Text"].astype(str).str.strip()
df = df[df["Review.Text"] != ""].copy()

print("\nMissing Values after cleaning:")
print(df.isnull().sum())


### **Exploratory Data Analysis**

EDA helps us understand the customer-review data before applying Generative AI. In this section, we examine numerical summaries, categorical diversity, product-rating distribution, department-level rating performance, and common words in highly-rated versus poorly-rated reviews.


In [ ]:
# Summary statistics for all numerical columns
numeric_summary = df.select_dtypes(include=np.number).describe().T.round(2)
display(numeric_summary)

# Optional distribution plots for the three most important numeric columns
fig_age = px.histogram(df, x="Age", nbins=35, title="Distribution of Customer Age")
fig_age.update_layout(xaxis_title="Age", yaxis_title="Number of Reviews")
fig_age.show()

fig_rating = px.histogram(df, x="Rating", color="Rating", title="Distribution of Product Ratings")
fig_rating.update_layout(xaxis_title="Rating (1-5)", yaxis_title="Number of Reviews")
fig_rating.show()

fig_feedback = px.histogram(
    df,
    x="Positive.Feedback.Count",
    nbins=60,
    log_y=True,
    title="Distribution of Positive Feedback Count (Log-Scaled Y Axis)"
)
fig_feedback.update_layout(xaxis_title="Positive Feedback Count", yaxis_title="Number of Reviews (log scale)")
fig_feedback.show()


**Numerical Summary Interpretation**

* `Age` ranges from 18 to 99, with an average of about 43 years and a median of 41 years. Most reviewers are concentrated between the mid-30s and early-50s, so the age distribution is moderately right-skewed because a smaller number of older customers extend the upper tail.
* `Rating` has a mean of about 4.18 and a median of 5. The 25th percentile is already 4, which shows that most customers gave high ratings. The rating distribution is therefore negatively skewed in the statistical sense, with most observations at the high end and a smaller tail of low scores.
* `Positive.Feedback.Count` has a mean of about 2.63 but a median of only 1, with a maximum of 122. This indicates a strongly right-skewed distribution: most reviews receive little or no positive feedback, while a small number of reviews are found helpful by many other customers.


In [ ]:
# Unique values in key categorical columns
categorical_cols = ["Division.Name", "Department.Name", "Class.Name"]

unique_counts = (
    df[categorical_cols]
    .nunique(dropna=True)
    .reset_index()
    .rename(columns={"index": "Categorical Column", 0: "Unique Values"})
)
display(unique_counts)

# Show the actual category values and review counts for each column
for col in categorical_cols:
    print(f"\n{col}: {df[col].nunique(dropna=True)} unique values")
    category_counts = df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Review Count")
    display(category_counts)


**Categorical Column Interpretation**

The cleaned dataset contains 3 unique divisions, 6 unique departments, and 20 unique product classes. This gives enough category variety to compare broad business areas such as Tops, Dresses, Bottoms, Intimate, Jackets, and Trend, while the `Class.Name` field provides a more granular product view for detailed merchandising analysis.


In [ ]:
# Overall distribution of product ratings
rating_distribution = df["Rating"].value_counts().sort_index().reset_index()
rating_distribution.columns = ["Rating", "Review Count"]
rating_distribution["Percentage"] = (rating_distribution["Review Count"] / len(df) * 100).round(2)
display(rating_distribution)

fig = px.bar(
    rating_distribution,
    x="Rating",
    y="Review Count",
    text="Percentage",
    color="Rating",
    title="Overall Distribution of Product Ratings"
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(xaxis_title="Rating (1-5)", yaxis_title="Number of Reviews")
fig.show()

positive_share = rating_distribution.loc[rating_distribution["Rating"].isin([4, 5]), "Review Count"].sum() / len(df) * 100
negative_share = rating_distribution.loc[rating_distribution["Rating"].isin([1, 2]), "Review Count"].sum() / len(df) * 100
print(f"Positive reviews (4-5 stars): {positive_share:.2f}%")
print(f"Negative reviews (1-2 stars): {negative_share:.2f}%")


**Rating Distribution Interpretation**

The dataset is strongly skewed toward positive reviews. About 55% of reviews are 5-star ratings and about 77% are either 4 or 5 stars. Only about 10% of reviews are 1 or 2 stars. This suggests customers are generally satisfied, but the lower-rated reviews remain especially important because they reveal specific pain points around fit, quality, color accuracy, or expectations.


In [ ]:
# Average rating by Department Name
dept_rating = (
    df.groupby("Department.Name")
    .agg(Review_Count=("Rating", "count"), Average_Rating=("Rating", "mean"))
    .sort_values("Average_Rating", ascending=False)
    .reset_index()
)
dept_rating["Average_Rating"] = dept_rating["Average_Rating"].round(3)
display(dept_rating)

highest_dept = dept_rating.iloc[0]
lowest_dept = dept_rating.iloc[-1]
print(f"Highest average rating: {highest_dept['Department.Name']} ({highest_dept['Average_Rating']:.3f})")
print(f"Lowest average rating: {lowest_dept['Department.Name']} ({lowest_dept['Average_Rating']:.3f})")

fig = px.bar(
    dept_rating.sort_values("Average_Rating"),
    x="Average_Rating",
    y="Department.Name",
    orientation="h",
    text="Average_Rating",
    color="Average_Rating",
    title="Average Customer Rating by Department"
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_title="Average Rating", yaxis_title="Department")
fig.show()


**Department Rating Interpretation**

`Bottoms` receives the highest average rating at about 4.279, followed closely by `Intimate` and `Jackets`. `Trend` receives the lowest average rating at about 3.839, although it also has far fewer reviews than the large departments. This may indicate that Bottoms are meeting customer expectations on fit and usability more consistently, while Trend items may involve higher style risk, expectation mismatch, or more variable quality. Because Trend has a small sample size, it should be monitored but interpreted with caution.


In [ ]:
# Word clouds and top words for highly-rated versus poorly-rated reviews
from sklearn.feature_extraction.text import CountVectorizer

positive_reviews = " ".join(df[df["Rating"].isin([4, 5])]["Review.Text"].astype(str))
negative_reviews = " ".join(df[df["Rating"].isin([1, 2])]["Review.Text"].astype(str))

wordcloud_positive = WordCloud(width=900, height=450, background_color="white", collocations=False).generate(positive_reviews)
wordcloud_negative = WordCloud(width=900, height=450, background_color="black", colormap="Reds", collocations=False).generate(negative_reviews)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))
ax1.imshow(wordcloud_positive, interpolation="bilinear")
ax1.set_title("Most Common Words in Highly-Rated Reviews (4-5 Stars)", fontsize=18)
ax1.axis("off")

ax2.imshow(wordcloud_negative, interpolation="bilinear")
ax2.set_title("Most Common Words in Poorly-Rated Reviews (1-2 Stars)", fontsize=18)
ax2.axis("off")
plt.show()

# Add a frequency table so the word-cloud insight is backed by exact counts.
def top_terms(text_series, n=20):
    vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=10)
    term_matrix = vectorizer.fit_transform(text_series.astype(str))
    frequencies = np.asarray(term_matrix.sum(axis=0)).ravel()
    terms = vectorizer.get_feature_names_out()
    return (
        pd.DataFrame({"Term": terms, "Frequency": frequencies})
        .sort_values("Frequency", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )

print("Top terms in highly-rated reviews (4-5 stars):")
display(top_terms(df[df["Rating"].isin([4, 5])]["Review.Text"]))

print("Top terms in poorly-rated reviews (1-2 stars):")
display(top_terms(df[df["Rating"].isin([1, 2])]["Review.Text"]))


**Word Cloud Interpretation and Initial Hypotheses**

Highly-rated reviews tend to contain words and phrases related to positive fit, emotional satisfaction, and styling appeal, such as love, great, perfect, fit, comfortable, flattering, size, dress, and wear. This suggests that customer satisfaction is driven by items that fit well, feel comfortable, look attractive, and match the customer's style expectations.

Poorly-rated reviews more often emphasize disappointment and mismatch, with recurring terms around fit, size, small, fabric, looked, quality, return, and wanted. This suggests dissatisfaction is often driven by sizing problems, poor fit, fabric or construction quality concerns, and gaps between product expectations and the actual item received.

An initial business hypothesis is that ChicStyle can improve customer satisfaction by strengthening size guidance, product images/descriptions, and fabric-quality controls. Fit and product expectation management appear to be the most important levers for reducing negative reviews.


## **Building the Generative AI Pipeline**

We will now build a system to analyze the reviews. This involves setting up the AI client, designing prompts, generating structured data, and evaluating the results. Use `gpt-4o-mini` model.

#### **Setup AI Client and Data Sample**

**Question: How do you initialize the OpenAI client with your API key and the correct base URL?**

In Google Colab, store the API key in Colab secrets as `OPENAI_API_KEY`, retrieve it with `userdata.get("OPENAI_API_KEY")`, and pass it to the OpenAI client along with the Great Learning base URL. The model name should be stored once in `model_name` so every later cell uses the same model consistently.


In [ ]:
# --- OpenAI API Setup ---
# Fetch your stored API key.
api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client. Provide your API key and the custom base URL.
client = openai.OpenAI(
    api_key=api_key,
    base_url="https://aibe.mygreatlearning.com/openai/v1"
)

model_name = "gpt-4o-mini"


#### **Note:**

For this project, we will analyze and categorize a sample of **50 customer reviews**. This number is chosen intentionally. Since the API has a **budget limit of $20**, running prompts on very large datasets can quickly exhaust your quota—especially because this exercise may involve **multiple iterations, prompt refinements, and repeated evaluations**.

To avoid unnecessary cost and ensure efficient experimentation, we recommend the following approach:

*   **Use very small samples (5–10 reviews)** during the **initial testing phase** to validate your prompt structure and logic.
    
*   **Scale up to 50 reviews** for the **final evaluation phase**, ensuring you get enough data to compare prompting techniques without draining your budget.
    
*   This strategy helps maintain cost control while still providing meaningful insights across Zero-Shot, Few-Shot, and Chain-of-Thought techniques.
    

If your API quota gets exhausted, you may temporarily switch to another Free AI assistant API. However, note that external tools may also have **rate limits** or **token caps**, so you will need to build retry logic and manage throttling within your code.

In [ ]:
# Create a sample of 50 reviews to work with.
# Reset the index so generated prediction DataFrames align correctly during concat.
df_sample_copy = df.sample(n=50, random_state=42).reset_index(drop=True)


#### **Prompt Engineering and Evaluation**

We will test three different prompting techniques. For each, we will create a basic version (V1) and an enhanced version (V2).

**Think about it:** A consistent evaluation framework is important because prompt outputs can vary in style, detail, and usefulness. Without a common rubric, comparisons across Zero-Shot, Few-Shot, and Chain-of-Thought prompts become subjective. An LLM-as-Judge can apply the same scoring criteria to every generated output and return a numeric score between 0 and 1 for relevance, sentiment correctness, structure, personalization, and business usefulness. The judge should be kept deterministic with `temperature=0` and should be instructed to return only a score.

#### **Technique 1: Zero-Shot Prompting**

**Answers:**

1. A basic Zero-Shot prompt should directly ask the model to analyze the review and return exactly five labeled fields: `Category`, `Sentiment`, `Summary`, `Personalized Message`, and `Retail Insight`. The output format must be strict so the parser can extract the fields.

2. A stronger V2 Zero-Shot prompt adds business context: ChicStyle is handling high-volume holiday feedback, accuracy matters for customer trust, and urgent product issues should be surfaced clearly. This helps the model produce more operationally useful summaries and insights.

3. The notebook loops through the 50-review sample with `tqdm`, calls `generate_output(review, prompt)` for each review, converts the list of dictionaries into a DataFrame, prefixes the columns by prompt version, and concatenates the results back to `df_sample`.

4. The LLM-as-Judge receives each review plus the generated structured output, applies the same rubric, returns a score from 0 to 1, and the average score is calculated for each prompt version.

**How the process works:**

1. First, create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.

2. Run **Zero-Shot Prompt Version 1** on a sample of 50 reviews to generate predictions.

3. Use the judge function to **score each prediction** and compute the **average score for Version 1**.

4. Repeat the same workflow with **Version 2**, evaluate it, and compare the average score with Version 1.


In [ ]:
# --- Generating LLM Predictions ---
def get_structured_output(review_text: str, prompt: str) -> dict:
    empty_output = {
        "Category": "",
        "Sentiment": "",
        "Summary": "",
        "Personalized_Message": "",
        "Retail_Insight": ""
    }

    try:
        # Make an API call to the chat completions endpoint.
        # Pass the model name, temperature, and messages (system and user roles).
        resp = client.chat.completions.create(
            model=model_name,
            temperature=0.0,
            max_tokens=350,
            messages=[
                {"role": "system", "content": "You are a precise retail feedback analyst. Return only the requested labeled fields."},
                {"role": "user", "content": f"{prompt}\n\nReview: {review_text}"}
            ]
        )
    except Exception as e:
        out = empty_output.copy()
        out["Summary"] = f"ERROR_CALL: {type(e).__name__}: {e}"
        out["Sentiment"] = "Error"
        return out

    text = resp.choices[0].message.content or ""
    out = empty_output.copy()

    # First try JSON, in case the model returns a dictionary-like response.
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            key_map = {
                "category": "Category",
                "sentiment": "Sentiment",
                "summary": "Summary",
                "personalized message": "Personalized_Message",
                "personalized_message": "Personalized_Message",
                "retail insight": "Retail_Insight",
                "retail_insight": "Retail_Insight"
            }
            for key, value in parsed.items():
                normalized_key = str(key).strip().lower().replace("-", " ")
                if normalized_key in key_map:
                    out[key_map[normalized_key]] = str(value).strip()
            if any(out.values()):
                return out
    except Exception:
        pass

    # Fallback parser for the required line-by-line format.
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]
    for line in lines:
        line = re.sub(r"^[\-\*\d\.\)\s]+", "", line)
        if ":" in line:
            key_raw, val = line.split(":", 1)
            key = key_raw.strip().lower().replace("_", " ")
            val = val.strip()
            if key.startswith("category"):
                out["Category"] = val
            elif key.startswith("sentiment"):
                out["Sentiment"] = val
            elif key.startswith("summary"):
                out["Summary"] = val
            elif "personal" in key:
                out["Personalized_Message"] = val
            elif "insight" in key:
                out["Retail_Insight"] = val
    return out


In [ ]:
# --- Prompting Strategies and Evaluation Framework ---
def generate_output(review_text, prompt):
    return get_structured_output(review_text, prompt)


def format_for_judge(row, prefix):
    """Package the original review and structured output so the judge can compare them."""
    return f"""
Review:
{row.get('Review.Text', '')}

Generated Output:
Category: {row.get(prefix + '_Category', '')}
Sentiment: {row.get(prefix + '_Sentiment', '')}
Summary: {row.get(prefix + '_Summary', '')}
Personalized Message: {row.get(prefix + '_Personalized_Message', '')}
Retail Insight: {row.get(prefix + '_Retail_Insight', '')}
""".strip()

# The judge_prompt contains the rules for the LLM evaluator.
judge_prompt = """
You are an impartial evaluator for a retail feedback intelligence system.
Score the generated output against the original review.

Rubric:
- 0.20 correct sentiment and tone interpretation
- 0.20 useful business category
- 0.20 accurate and concise summary
- 0.20 empathetic, specific personalized message
- 0.20 actionable retail insight

Return only one decimal number from 0 to 1. Do not include words, bullets, or explanations.

{output}
"""

def judge_output(output_text):
    try:
        # Call the LLM with the judge_prompt to get a score.
        resp = client.chat.completions.create(
            model=model_name,
            temperature=0,
            max_tokens=10,
            messages=[
                {"role": "system", "content": "You are a strict evaluator that returns only a numeric score."},
                {"role": "user", "content": judge_prompt.format(output=output_text)}
            ]
        )
        score_str = resp.choices[0].message.content.strip()
        match = re.search(r"0(?:\.\d+)?|1(?:\.0+)?", score_str)
        if not match:
            return None
        return max(0.0, min(1.0, float(match.group(0))))
    except Exception:
        return None


In [ ]:
print("Zero-Shot Version 1 predictions...")
df_sample = df_sample_copy.copy()

# Define the first version of your Zero-Shot prompt.
zero_shot_prompt_v1 = """Analyze the customer review and return exactly five lines in this format:
Category: <main issue or praise area>
Sentiment: <Positive, Negative, or Neutral>
Summary: <one concise sentence>
Personalized Message: <one empathetic customer-facing sentence>
Retail Insight: <one actionable business insight>"""


# Loop through each review in the sample, call the generate_output function, and store the results.
zs_list = [generate_output(review, zero_shot_prompt_v1) for review in tqdm(df_sample["Review.Text"], desc="Zero-Shot Predictions")]
zs_df = pd.DataFrame(zs_list)
df_sample = pd.concat([df_sample, zs_df.add_prefix("ZS_V1_")], axis=1)


In [ ]:
print("Judging Zero-Shot V1 outputs...")
# Loop through the generated outputs and use the judge_output function to score them.
zs_scores = [
    judge_output(format_for_judge(row, "ZS_V1"))
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Judge ZS_V1")
]
df_sample["ZS_V1_Score"] = zs_scores
print(f"Avg ZS_V1_Score: {df_sample['ZS_V1_Score'].mean():.3f}")


In [ ]:
print("\nZero-Shot V2 predictions (context-enhanced)...")

# Define the second version of your Zero-Shot prompt.
zero_shot_prompt_v2 = """You are analyzing real-time holiday-season reviews for ChicStyle, a fashion retail platform. Accuracy matters because delayed or incorrect review handling can damage customer trust during peak sales periods.

Classify the review using retail business context. Prioritize issues related to fit, sizing, product quality, comfort, color/image mismatch, delivery/service, and value. If a review contains mixed feedback, choose Neutral unless the overall tone clearly leans positive or negative.

Return exactly five lines and no extra commentary:
Category: <main issue or praise area>
Sentiment: <Positive, Negative, or Neutral>
Summary: <one concise sentence grounded in the review>
Personalized Message: <one empathetic customer-facing sentence>
Retail Insight: <one action ChicStyle can take or monitor>"""

# Loop through each review in the sample, call the generate_output function, and store the results.
zsv2_list = [generate_output(review, zero_shot_prompt_v2) for review in tqdm(df_sample["Review.Text"], desc="Zero-Shot V2 Predictions")]
zsv2_df = pd.DataFrame(zsv2_list)
df_sample = pd.concat([df_sample, zsv2_df.add_prefix("ZS_V2_")], axis=1)


In [ ]:
print("Judging Zero-Shot V2 outputs...")
# Loop through the generated outputs and use the judge_output function to score them.
zsv2_scores = [
    judge_output(format_for_judge(row, "ZS_V2"))
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Judge ZS_V2")
]
df_sample["ZS_V2_Score"] = zsv2_scores
print(f"Avg ZS_V2_Score: {df_sample['ZS_V2_Score'].mean():.3f}")


#### **Technique 2: Few-Shot Prompting**

**Answers:**

1. A Few-Shot prompt should include the task instructions plus a small set of high-quality examples showing the desired input-output pattern. The best examples are representative and contrasting: at least one clearly positive review, one clearly negative review, and one mixed/neutral review. This teaches the model both the label set and the exact output structure.

2. The V2 prompt can add explicit rules for each field: choose one dominant category, use only Positive/Negative/Neutral for sentiment, keep the summary factual, make the personalized message empathetic but not overpromising, and make the retail insight actionable for merchandising, product, or customer experience teams.

3. Few-Shot prompting should generally outperform the basic Zero-Shot prompt because examples reduce ambiguity and improve format consistency. Few-Shot V2 is expected to be stronger than Few-Shot V1 because the examples are supported by business rules and clearer constraints.

**How the process works:**

1. First, create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.

2. Run **Few-Shot Prompt Version 1** on a sample of 50 reviews to generate predictions.

3. Use the judge function to **score each prediction** and compute the **average score for Version 1**.

4. Repeat the same workflow with **Version 2**, evaluate it, and compare the average score with previous versions.


In [ ]:
# --- Technique 2: Few-Shot Prompting ---

print("\nFew-Shot V1 predictions...")
# Define the first version of your Few-Shot prompt.
few_shot_prompt_v1 = """Analyze the customer review and return the same five fields shown in the examples.

Example 1
Review: Love this dress. It fits perfectly and the fabric feels soft.
Category: Fit and Comfort
Sentiment: Positive
Summary: The customer loves the dress because it fits well and feels comfortable.
Personalized Message: Thank you for sharing your experience; we are glad the fit and fabric worked so well for you.
Retail Insight: Fit and fabric comfort are strong selling points for this item.

Example 2
Review: I wanted to love this top, but it ran very small and the fabric felt cheap.
Category: Fit/Sizing and Product Quality
Sentiment: Negative
Summary: The customer is disappointed because the item ran small and the fabric quality felt poor.
Personalized Message: We are sorry the top did not meet expectations and appreciate the feedback on sizing and fabric.
Retail Insight: Review sizing guidance and fabric quality for this product.

Now analyze the new review. Return exactly five lines:
Category: <main issue or praise area>
Sentiment: <Positive, Negative, or Neutral>
Summary: <one concise sentence>
Personalized Message: <one empathetic customer-facing sentence>
Retail Insight: <one actionable business insight>"""

# Loop through each review in the sample, call the generate_output function, and store the results.
fs_list = [generate_output(review, few_shot_prompt_v1) for review in tqdm(df_sample["Review.Text"], desc="Few-Shot V1 Predictions")]
fs_df = pd.DataFrame(fs_list)
df_sample = pd.concat([df_sample, fs_df.add_prefix("FS_V1_")], axis=1)


In [ ]:
print("Judging Few-Shot V1 outputs...")
# Loop through the generated outputs and use the judge_output function to score them.
fs_v1_scores = [
    judge_output(format_for_judge(row, "FS_V1"))
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Judge FS_V1")
]
df_sample["FS_V1_Score"] = fs_v1_scores
print(f"Avg FS_V1_Score: {df_sample['FS_V1_Score'].mean():.3f}")


In [ ]:
print("\nFew-Shot V2 predictions (context-enhanced)...")
# Define the second version of your few-Shot prompt.
few_shot_prompt_v2 = """You are ChicStyle's retail feedback analyst. Use the examples and rules below to produce consistent structured outputs.

Rules:
1. Category must be specific, such as Fit/Sizing, Product Quality, Style/Design, Comfort, Color/Image Accuracy, Delivery/Service, Price/Value, or General Experience.
2. Sentiment must be exactly Positive, Negative, or Neutral. Use Neutral for mixed reviews with balanced praise and criticism.
3. Summary must be factual and no more than 25 words.
4. Personalized Message must acknowledge the customer's exact experience and avoid offering refunds or policies not stated in the review.
5. Retail Insight must be an action or monitoring recommendation for the business.

Example 1
Review: This jacket is beautiful, warm, and true to size. I received many compliments.
Category: Style/Design and Fit/Sizing
Sentiment: Positive
Summary: The customer liked the jacket's style, warmth, and true-to-size fit.
Personalized Message: We are delighted that the jacket fit well and brought you compliments.
Retail Insight: Highlight warmth, fit accuracy, and styling in product messaging.

Example 2
Review: The pants looked nothing like the photo and were too tight in the waist.
Category: Color/Image Accuracy and Fit/Sizing
Sentiment: Negative
Summary: The customer felt the pants did not match the photo and were too tight.
Personalized Message: We are sorry the pants did not match your expectations in appearance or fit.
Retail Insight: Audit product photos and waist-fit guidance for this item.

Example 3
Review: The sweater is cute and soft, but the sleeves stretched out after one wear.
Category: Product Quality and Comfort
Sentiment: Neutral
Summary: The customer liked the sweater's look and softness but noticed sleeve stretching after one wear.
Personalized Message: Thank you for noting both the comfort and the sleeve durability concern.
Retail Insight: Investigate sleeve recovery and fabric durability while preserving softness.

Now analyze the new review. Return exactly five lines and no extra text:
Category: <main issue or praise area>
Sentiment: <Positive, Negative, or Neutral>
Summary: <one concise sentence>
Personalized Message: <one empathetic customer-facing sentence>
Retail Insight: <one actionable business insight>"""

# Loop through each review in the sample, call the generate_output function, and store the results.
fs_v2_list = [generate_output(review, few_shot_prompt_v2) for review in tqdm(df_sample["Review.Text"], desc="Few-Shot V2 Predictions")]
fs_v2_df = pd.DataFrame(fs_v2_list)
df_sample = pd.concat([df_sample, fs_v2_df.add_prefix("FS_V2_")], axis=1)


In [ ]:
print("Judging Few-Shot V2 outputs...")
# Loop through the generated outputs and use the judge_output function to score them.
fs_v2_scores = [
    judge_output(format_for_judge(row, "FS_V2"))
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Judge FS_V2")
]
df_sample["FS_V2_Score"] = fs_v2_scores
print(f"Avg FS_V2_Score: {df_sample['FS_V2_Score'].mean():.3f}")


#### **Technique 3: Chain-of-Thought (CoT) Prompting**

**Answers:**

1. The prompt should instruct the model to reason internally before answering, but it should also explicitly say: "Do not show your reasoning. Return only the final structured answer." This gets the benefit of deliberate analysis without exposing verbose reasoning or breaking the parser.

2. CoT V2 can combine internal reasoning with the stronger business context and field-level rules from the enhanced prompts. The model can silently check the review for sentiment, main issue, urgency, customer impact, and business action before returning the five required lines.

3. Encouraging internal reasoning can improve output quality when reviews contain mixed or subtle feedback. The measurable improvement should be checked using the average LLM-as-Judge scores. CoT V2 is expected to be the most reliable when it follows the strict output format because it combines context, rules, and internal reasoning.

**How the process works:**

1. First, create an **LLM-as-a-judge** function that can evaluate the quality of model outputs.

2. Run **CoT Prompt Version 1** on a sample of 50 reviews to generate predictions.

3. Use the judge function to **score each prediction** and compute the **average score for Version 1**.

4. Repeat the same workflow with **Version 2**, evaluate it, and compare the average score with other prompting techniques.


In [ ]:
# --- Technique 3: Chain-of-Thought (CoT) Prompting ---
print("\nCoT V1 predictions...")

# Define the first version of your COT prompt.
cot_prompt_v1 = """Analyze the review by thinking through the customer's main issue, sentiment, and business implication internally. Do not reveal your reasoning.

Return only the final answer in exactly five lines:
Category: <main issue or praise area>
Sentiment: <Positive, Negative, or Neutral>
Summary: <one concise sentence>
Personalized Message: <one empathetic customer-facing sentence>
Retail Insight: <one actionable business insight>"""

# Loop through each review in the sample, call the generate_output function, and store the results.
cot_list = [generate_output(review, cot_prompt_v1) for review in tqdm(df_sample["Review.Text"], desc="CoT V1 Predictions")]
cot_df = pd.DataFrame(cot_list)
df_sample = pd.concat([df_sample, cot_df.add_prefix("COT_V1_")], axis=1)


In [ ]:
print("Judging CoT V1 outputs...")
# Loop through the generated outputs and use the judge_output function to score them.
cot_v1_scores = [
    judge_output(format_for_judge(row, "COT_V1"))
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Judge COT_V1")
]
df_sample["COT_V1_Score"] = cot_v1_scores
print(f"Avg COT_V1_Score: {df_sample['COT_V1_Score'].mean():.3f}")


In [ ]:
print("\nCoT V2 predictions (context-enhanced)...")
# Define the second version of your COT prompt.
cot_prompt_v2 = """You are ChicStyle's real-time retail feedback analyst during a high-volume holiday sales period.

Before answering, reason internally through these steps:
1. Identify whether the review is mainly about fit/sizing, quality, style/design, comfort, color/image accuracy, delivery/service, price/value, or general experience.
2. Determine whether the overall sentiment is Positive, Negative, or Neutral, treating balanced mixed feedback as Neutral.
3. Separate the customer's emotional concern from the operational business issue.
4. Decide what the customer support team should acknowledge and what the merchandising/product team should learn.

Do not show the reasoning. Return exactly five lines and no extra commentary:
Category: <specific business category>
Sentiment: <Positive, Negative, or Neutral>
Summary: <one factual sentence of 25 words or fewer>
Personalized Message: <one empathetic customer-facing sentence>
Retail Insight: <one actionable business insight for ChicStyle>"""

# Loop through each review in the sample, call the generate_output function, and store the results.
cot_v2_list = [generate_output(review, cot_prompt_v2) for review in tqdm(df_sample["Review.Text"], desc="CoT V2 Predictions")]
cot_v2_df = pd.DataFrame(cot_v2_list)
df_sample = pd.concat([df_sample, cot_v2_df.add_prefix("COT_V2_")], axis=1)


In [ ]:
print("Judging CoT V2 outputs...")
# Loop through the generated outputs and use the judge_output function to score them.
cot_v2_scores = [
    judge_output(format_for_judge(row, "COT_V2"))
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc="Judge COT_V2")
]
df_sample["COT_V2_Score"] = cot_v2_scores
print(f"Avg COT_V2_Score: {df_sample['COT_V2_Score'].mean():.3f}")


## **Applying GenAI for Product Recommendation:**

Now, let's use the model for a different task: predicting the `Recommended.IND` flag.

**Answers:**

1. The recommendation prompt should be strict and short. It should tell the model to use only the review text and return exactly two lines: `Recommended: 1` or `Recommended: 0`, followed by `Reason: <brief reason>`. This reduces parsing errors and avoids long narrative responses.

2. A regex-based parsing function is needed to reliably extract the binary flag and reason even if the model adds small formatting variations. The parser should return `(flag, reason, raw_text)` and use `np.nan` if no valid 0/1 flag is found.

3. The predictions should be compared with the original human label using standard classification metrics: `accuracy_score` for overall correctness, `confusion_matrix` for false positives/false negatives, and `classification_report` for precision, recall, and F1-score.

**How the Process Works**

**1. Prepare Data**

Copy the dataset, store the original recommendation labels, and remove them from the model input to avoid leakage.

**2. Generate Predictions**

Use a strict two-line prompt to make the LLM output a binary recommendation (1/0) and a short reason based only on the review text.

**3. Parse Outputs**

Extract the flag and reason from the raw LLM response using regex-based parsing that handles formatting issues.

**4. Build Prediction Table**

Run the prompt for each review, parse the result, and store the predictions in a new DataFrame.

**5. Evaluate Performance**

Compare LLM predictions with true labels using accuracy, confusion matrix, and classification report.

**6. Explain Mismatches**

For incorrect predictions, generate a short explanation describing why the model's decision may have differed from the human label.


In [ ]:
# --- Predicting Product Recommendations ---
df_reco = df_sample.copy()
df_reco["Actual_Recommended"] = df_reco["Recommended.IND"]
df_reco = df_reco.drop(columns=["Recommended.IND"])

In [ ]:
# The prompt and parsing function are provided for you.
recommendation_prompt = """
Predict whether the customer would recommend the product based only on the review text.

Return exactly two lines:
Recommended: <1 if the customer would recommend, otherwise 0>
Reason: <brief reason using 20 words or fewer>

Rules:
- Use 1 for clearly positive or recommendable experiences.
- Use 0 for clearly negative experiences, returns, poor fit, defects, misleading photos, or strong dissatisfaction.
- For mixed reviews, choose the label that best matches the customer's overall likelihood to recommend.
- Do not include any extra text.
"""


In [ ]:
def parse_recommendation_reply(raw_text):
    if raw_text is None: return (np.nan, "", "")
    s = str(raw_text).strip()
    m = re.search(r'Recommended\s*:\s*([01])\b', s, re.IGNORECASE)
    flag = int(m.group(1)) if m else np.nan
    m3 = re.search(r'Reason\s*:\s*(.+)', s, re.IGNORECASE)
    reason = m3.group(1).strip() if m3 else ""
    return (flag, reason[:200], s)

llm_preds = []
for review in tqdm(df_reco["Review.Text"], desc="LLM Recommend Predictions"):
    prompt = recommendation_prompt + "\n\nReview:\n" + (review or "")
    try:
        # Call the LLM to get a recommendation prediction (1 or 0).
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role":"user", "content": prompt}],
            temperature=0.0, max_tokens=60
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        raw = f"ERROR: {type(e).__name__}: {e}"

    # Parse the raw response to extract the flag and reason.
    flag, reason, raw_clean = parse_recommendation_reply(raw)
    llm_preds.append({"LLM_Recommended_Raw": raw_clean, "LLM_Recommended_Flag": flag, "LLM_Recommend_Reason": reason})

llm_pred_df = pd.DataFrame(llm_preds)
df_reco = pd.concat([df_reco.reset_index(drop=True), llm_pred_df.reset_index(drop=True)], axis=1)


In [ ]:
# --- Evaluating Model Performance ---
eval_df = df_reco.dropna(subset=["LLM_Recommended_Flag"]).copy()
if not eval_df.empty:
    y_true = eval_df["Actual_Recommended"].astype(int)
    y_pred = eval_df["LLM_Recommended_Flag"].astype(int)

    # Calculate the accuracy score.
    acc = accuracy_score(y_true, y_pred)
    # Generate the confusion matrix.
    cm = confusion_matrix(y_true, y_pred)
    # Generate the classification report.
    report = classification_report(y_true, y_pred, digits=3)

    print(f"Accuracy: {acc:.3f}\n")
    print("Confusion Matrix:\n", cm)
    print("\nClassification report:\n", report)
else:
    print("No valid LLM recommendation predictions were available for evaluation.")


In [ ]:
# This variable stores the explanation prompt template.
# Later, we fill in {human_label} and {llm_label} for each mismatched row.
explain_prompt = """
The human recommendation label is {human_label}, but the LLM predicted {llm_label}.
Explain the likely reason for this mismatch in one concise sentence.
Focus on ambiguity, mixed sentiment, missing context, or wording in the review.
"""

# -------------------------------------------------------------------------
# STEP 1: Filter all rows where the model's prediction does NOT match the
#         actual human label.
# -------------------------------------------------------------------------
diff_rows = df_reco[
    (df_reco["LLM_Recommended_Flag"].notna()) &
    (df_reco["LLM_Recommended_Flag"].astype(float) != df_reco["Actual_Recommended"].astype(float))
]

# This list will hold the explanation text returned by the LLM for each mismatch.
explanations = []

# -------------------------------------------------------------------------
# STEP 2: Loop through each mismatched row and generate an explanation
# -------------------------------------------------------------------------
# NOTE: iterrows() is used here because the dataset has dotted column names
# such as Review.Text, which are renamed unpredictably by itertuples().
# -------------------------------------------------------------------------
for idx, row in tqdm(diff_rows.iterrows(), total=len(diff_rows), desc="Explain mismatches"):

    # Safely extract review text.
    review = row.get("Review.Text", "") or ""

    # Convert values to integers to avoid type-related issues.
    llm_label = int(row.get("LLM_Recommended_Flag"))
    human_label = int(row.get("Actual_Recommended"))

    # Build the final prompt by inserting values into the template.
    prompt = explain_prompt.format(
        human_label=human_label,
        llm_label=llm_label
    ) + "\n\nReview:\n" + review

    try:
        # -----------------------------------------------------------------
        # STEP 3: Send prompt to the LLM model to generate the explanation
        # -----------------------------------------------------------------
        resp = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a concise analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=40
        )

        # Extract the text response from the LLM safely.
        choice = resp.choices[0]
        msg = getattr(choice, "message", None)

        if msg is not None:
            raw = msg.get("content", "") if isinstance(msg, dict) else getattr(msg, "content", "") or ""
        elif hasattr(choice, "text"):
            raw = getattr(choice, "text", "") or ""
        else:
            raw = str(choice)

        # Clean and trim the explanation text.
        raw = raw.strip().replace("\n", " ")
        explanations.append(raw[:250])

    except Exception as e:
        explanations.append(f"ERROR: {type(e).__name__}")

# -------------------------------------------------------------------------
# STEP 4: Add a new column to df_reco for storing explanations
# -------------------------------------------------------------------------
df_reco["LLM_Diff_Explain"] = ""

# -------------------------------------------------------------------------
# STEP 5: Attach each explanation to the correct row using index alignment
# -------------------------------------------------------------------------
for i, idx in enumerate(diff_rows.index):
    df_reco.at[idx, "LLM_Diff_Explain"] = explanations[i]

# Final confirmation message.
print(f"Added explanations for {len(diff_rows)} mismatches.")

df_reco


**Visualization of Sentiments Distribution**

After generating results from all prompting techniques, it is crucial to visualize their sentiment outputs to understand whether the methods behave similarly or differently.

**Answers:**

* The V2 sentiment distributions should be compared by plotting the counts of Positive, Negative, and Neutral predictions for Zero-Shot V2, Few-Shot V2, and CoT V2. If all three methods are stable, their distributions should be broadly similar, with differences mostly around mixed reviews.

* Noticeable differences are meaningful. A method that predicts many more Neutral reviews may be more cautious and better at recognizing mixed feedback, but it may also under-classify clearly positive or negative reviews. A method that predicts mostly Positive may be overly influenced by the positive skew in the dataset. The most reliable method is the one with strong judge scores and a reasonable sentiment distribution, not simply the one with the most positive predictions.


In [ ]:
# ----------------------------
# Helper function for bar chart
# ----------------------------
def normalize_sentiment(value):
    text = str(value).strip().lower()
    if "positive" in text:
        return "Positive"
    if "negative" in text:
        return "Negative"
    if "neutral" in text:
        return "Neutral"
    if text in {"", "nan", "none"}:
        return "Missing"
    return text.title()


def plot_sentiment(df, column_name, title):
    sentiment_counts = df[column_name].apply(normalize_sentiment).value_counts().reset_index()
    sentiment_counts.columns = ["Sentiment", "Count"]

    fig = px.bar(
        sentiment_counts,
        x="Sentiment",
        y="Count",
        text="Count",
        title=title,
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(xaxis_title="Sentiment", yaxis_title="Count")
    fig.show()


# ----------------------------
# Plot all three methods
# ----------------------------
plot_sentiment(df_sample, "ZS_V1_Sentiment", "Zero-Shot Sentiment Distribution for Version 1")
plot_sentiment(df_sample, "FS_V1_Sentiment", "Few-Shot Sentiment Distribution for Version 1")
plot_sentiment(df_sample, "COT_V1_Sentiment", "Chain-of-Thought Sentiment Distribution for Version 1")
plot_sentiment(df_sample, "ZS_V2_Sentiment", "Zero-Shot Sentiment Distribution for Version 2")
plot_sentiment(df_sample, "FS_V2_Sentiment", "Few-Shot Sentiment Distribution for Version 2")
plot_sentiment(df_sample, "COT_V2_Sentiment", "Chain-of-Thought Sentiment Distribution for Version 2")


##  **Comparison of Prompting Techniques:**

* Zero-Shot is the fastest and simplest approach. It can perform well when the review is clear, but it is more likely to produce inconsistent categories or less specific business insights.

* Few-Shot improves consistency because examples demonstrate the expected format and level of detail. Few-Shot V2 should be more reliable than V1 because the rules reduce ambiguity and make the output easier to parse.

* CoT prompting is most useful for mixed reviews where the model must separate praise from complaints. CoT V2 is expected to be the most reliable and consistent because it combines business context, explicit rules, and internal reasoning while still returning only the final structured answer.

* For production, I would propose `gpt-4o-mini` with the CoT V2-style prompt, strict output schema, low temperature, retry/error handling, validation of parsed fields, and periodic human review of low-confidence or high-impact complaints.


In [ ]:
# ---------------------------------------------------------
# Build comparison records for the LLM to analyze
# ---------------------------------------------------------

comparison_records = []
for idx, row in df_sample.iterrows():
    rec = {
        "Review": row.get("Review.Text", ""),
        "ZeroShot_Sentiment": row.get("ZS_V2_Sentiment", None),
        "FewShot_Sentiment": row.get("FS_V2_Sentiment", None),
        "CoT_Sentiment": row.get("COT_V2_Sentiment", None)
    }
    comparison_records.append(rec)
summary_text = json.dumps(comparison_records, indent=2)

score_columns = [
    "ZS_V1_Score", "ZS_V2_Score",
    "FS_V1_Score", "FS_V2_Score",
    "COT_V1_Score", "COT_V2_Score"
]
score_summary = df_sample[score_columns].apply(pd.to_numeric, errors="coerce").mean().dropna()
score_summary = {k: float(round(v, 3)) for k, v in score_summary.to_dict().items()}

compare_prompt = f"""
You are evaluating three prompting strategies for retail review intelligence.

Average LLM-as-Judge scores:
{json.dumps(score_summary, indent=2)}

V2 sentiment comparison records:
{summary_text}

Tasks:
1. Compare Zero-Shot V2, Few-Shot V2, and CoT V2 in terms of sentiment consistency and business usefulness.
2. Identify which technique appears most reliable and explain why.
3. Recommend the best model and prompt design for production.

Keep the answer concise but decision-ready.
"""

# Call the LLM with the compare_prompt to get a qualitative comparison of the techniques.
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are an expert in evaluating LLM prompting strategies."},
        {"role": "user", "content": compare_prompt}
    ],
    temperature=0
)
print(response.choices[0].message.content)


### **Observations and Insights**

**Refined Insights:**

Based on the dataset and the expected strongest prompt design, the most meaningful recurring customer themes are:

* Fit and sizing accuracy are central to customer satisfaction. Positive reviews often mention "true to size" or a perfect fit, while negative and neutral reviews frequently mention items running small, large, tight, or not working for the customer's body shape.

* Product quality and fabric expectations drive dissatisfaction when items feel cheap, thin, itchy, or lose shape after wearing. These quality signals are especially important because customers often wanted to like the item but rejected it after trying it.

* Style and visual appeal are strong positive drivers. Many favorable reviews praise dresses, tops, and jackets for looking flattering, receiving compliments, and matching the customer's desired style.

* Mixed feedback is common. Reviews may praise one aspect, such as design or comfort, while criticizing color accuracy, sizing, or construction. This is where the enhanced Few-Shot and CoT prompts add value over simple sentiment classification.

* Tops and dresses generate large review volume and a meaningful share of negative feedback, so they should be monitored closely during peak sales periods.


# Generating Actionable Product Improvement Suggestions

**Short-term recommendations (3-6 months):**

* Improve fit guidance by adding clearer size notes, model measurements, and review-derived fit tags such as "runs small," "true to size," or "relaxed fit."

* Create a rapid-response workflow for urgent negative reviews involving defects, misleading photos, fabric quality, or severe fit issues so support and merchandising teams can react quickly during peak sales.

* Audit high-volume categories, especially tops and dresses, for recurring complaints about sizing, fabric feel, color mismatch, and construction quality.

**Long-term recommendations (6-12 months):**

* Build a product feedback dashboard that aggregates sentiment, category, and insight trends by department, class, and product ID.

* Use review intelligence in buying and product development decisions, especially for fabric selection, size grading, color accuracy, and vendor quality control.

* Deploy a production GenAI review pipeline with strict schemas, monitoring, human escalation for high-risk complaints, and regular prompt evaluation against human-labeled samples.

**Business value:**

This automated GenAI pipeline solves the original business problem by turning large volumes of unstructured reviews into structured, actionable intelligence. It helps ChicStyle respond faster to customers, detect recurring product issues earlier, protect brand trust during high-pressure shopping seasons, and make better merchandising and product improvement decisions.


In [ ]:
score_columns = [
    "ZS_V1_Score", "ZS_V2_Score",
    "FS_V1_Score", "FS_V2_Score",
    "COT_V1_Score", "COT_V2_Score"
]
score_means = df_sample[score_columns].apply(pd.to_numeric, errors="coerce").mean().dropna()
score_table = {k: float(round(v, 3)) for k, v in score_means.to_dict().items()}

best_score_col = score_means.idxmax() if not score_means.empty else "COT_V2_Score"
best_prefix = best_score_col.replace("_Score", "")

best_sentiment_col = f"{best_prefix}_Sentiment"
best_summary_col = f"{best_prefix}_Summary"
best_insight_col = f"{best_prefix}_Retail_Insight"
selected_cols = [
    col for col in ["Review.Text", "Rating", "Department.Name", best_sentiment_col, best_summary_col, best_insight_col]
    if col in df_sample.columns
]
aggregated_records = df_sample[selected_cols].fillna("").astype(str).to_dict("records")

business_prompt = f"""
You are creating executive recommendations for ChicStyle based on customer review intelligence.

Best-performing prompt version selected by average judge score: {best_prefix}
Average judge scores:
{json.dumps(score_table, indent=2)}

Aggregated review insights from the selected model:
{json.dumps(aggregated_records, indent=2)}

Provide:
1. Three short-term actions for the next 3-6 months.
2. Three long-term actions for the next 6-12 months.
3. A concise explanation of how this GenAI pipeline creates business value.

Make every recommendation specific, actionable, and tied to retail operations.
"""

# Call the LLM with the business_prompt to generate strategic recommendations.
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a retail business strategist."},
        {"role": "user", "content": business_prompt}
    ],
    temperature=0,
    max_tokens=800
)
print(response.choices[0].message.content)


### **Observations and Insights**

The completed pipeline demonstrates how raw review text can be converted into structured intelligence for both customer support and business decision-making. The key analytical value is not only sentiment labeling, but the combination of category, summary, customer message, and retail insight. That combination makes the output usable by multiple teams: support can respond to customers, merchandising can identify recurring product issues, and leadership can prioritize improvements by category and urgency.

A strong production version should keep the strict schema, low temperature, validation checks, and human review for high-impact complaints. The model should also be monitored over time because customer language, product assortment, and seasonal pain points can change.


## **Conclusion**

This project builds a practical Generative AI workflow for real-time retail feedback intelligence. The notebook loads and cleans customer review data, explores rating and department patterns, generates structured review intelligence using Zero-Shot, Few-Shot, and Chain-of-Thought prompts, evaluates prompt quality with an LLM-as-Judge, predicts recommendation intent, and converts aggregated insights into business actions.

The best production approach is an enhanced, schema-constrained prompt using `gpt-4o-mini`, low temperature, robust parsing, validation, and periodic human review. This design helps ChicStyle respond faster during peak shopping periods, identify recurring product issues, personalize customer communication, and protect long-term customer loyalty.


Completed notebook.
